# Properties and Descriptors — Advanced Tutorial Problems, Volume II

### Fourteen **new** worked problems, developed one step at a time

This notebook follows the style of the supplied lesson: short explanations, small experiments, careful questions, and gradually assembled solutions. It explores **different advanced situations** rather than reproducing the earlier exercise set.

**How to use it:** run top to bottom. At each numbered step, read the explanatory Markdown, predict what its short code cell will do, then execute it. Every problem finishes with explicit assertions and an explanation of trade-offs. All examples are pure Python and need no third-party packages. Python 3.10+ is recommended. The snippets are demonstrations of normal Python behavior, not security boundaries.

## A small test helper

We will intentionally trigger a few exceptions to understand *why* descriptor access fails. Catch only the expected exception; this makes an unexpectedly successful operation fail the test instead of silently passing. The helper is shared by the entire tutorial.

In [1]:
from contextlib import contextmanager

@contextmanager
def expect_error(error_type, contains=None):
    try:
        yield
    except error_type as error:
        if contains is not None:
            assert contains in str(error), (type(error).__name__, str(error))
    else:
        raise AssertionError(f"Expected {error_type.__name__}")

print("Test helper ready.")

Test helper ready.


---

## Problem 01 — Property decorators create *new objects*

**Challenge.** Investigate what `@reading.setter` does to the property object, its getter, and its documentation. Explain why this matters when reusing a property in subclasses.

**Learning path.** Inspect a getter-only property; attach a setter; compare identities and function references; confirm runtime behavior.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Look at the starting object

A getter-only property is a descriptor stored on the class. We can inspect the descriptor without evaluating its getter by accessing it through the class.

In [2]:
def read_temperature(self):
    """Measured temperature in Celsius."""
    return self._temperature

readonly_temperature = property(read_temperature)
assert readonly_temperature.fget is read_temperature
assert readonly_temperature.fset is None
print(readonly_temperature.__doc__)

Measured temperature in Celsius.


**What this shows.** The `fget`, `fset`, and `__doc__` attributes reveal the property’s configuration. `fset is None` does **not** make the property a non-data descriptor: its descriptor type still implements a setter that rejects writes.

### Step 2 — Add a setter without mutating the old object

The `setter()` method returns another property. We deliberately keep references to both the original and the result so we can compare them.

In [3]:
def write_temperature(self, value):
    number = float(value)
    if number < -273.15:
        raise ValueError("below absolute zero")
    self._temperature = number

writable_temperature = readonly_temperature.setter(write_temperature)
assert writable_temperature is not readonly_temperature
assert readonly_temperature.fset is None
assert writable_temperature.fget is read_temperature
assert writable_temperature.fset is write_temperature
print("original and replacement are different:", writable_temperature is not readonly_temperature)

original and replacement are different: True


### Step 3 — Install the property in a class

Decorators perform the same replacement inside a class body. Here we install the already built property explicitly, so no mystery remains about what is stored in the class namespace.

In [4]:
class Thermometer01:
    temperature = writable_temperature

    def __init__(self, initial):
        self.temperature = initial

thermometer01 = Thermometer01(20)
assert thermometer01.temperature == 20.0
with expect_error(ValueError, "absolute zero"):
    thermometer01.temperature = -300
assert thermometer01.temperature == 20.0
print("temperature:", thermometer01.temperature)

temperature: 20.0


**What this shows.** The setter validates before changing `_temperature`, so the invalid write leaves the previous value unchanged. A decorator such as `@temperature.setter` is essentially class-body assignment to the new descriptor object.

---

## Problem 02 — A deleter can model a resettable state

**Challenge.** Write a managed property that can be assigned and deleted. Deleting must reset the state, not delete the whole object or expose an inconsistent attribute.

**Learning path.** Observe a property without a deleter; add a carefully defined deleter; test the entire assignment/deletion cycle.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Establish what deletion normally does

Properties implement deletion as part of the descriptor protocol, but without an `fdel` function they reject the operation.

In [5]:
class ImmutableToken02:
    @property
    def token(self):
        return "generated"

readonly02 = ImmutableToken02()
with expect_error(AttributeError):
    del readonly02.token
print("Deletion is not available by default.")

Deletion is not available by default.


### Step 2 — Decide on the invariant

For the new type, `None` means “not configured.” The getter will return that state; the setter accepts only nonempty strings; the deleter restores `None`. All three operations have explicit meaning.

In [6]:
class Session02:
    def __init__(self):
        self._token = None
        self.events = []

    @property
    def token(self):
        """Currently configured token, or None."""
        return self._token

    @token.setter
    def token(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("token must be a nonempty string")
        self._token = value
        self.events.append("assigned")

    @token.deleter
    def token(self):
        self._token = None
        self.events.append("reset")

### Step 3 — Exercise the state transitions

Notice that a failed validation must not append an event. Deletion is repeatable here because reset-to-`None` is defined to be idempotent.

In [7]:
session02 = Session02()
assert session02.token is None
session02.token = "abc"
with expect_error(ValueError):
    session02.token = "  "
assert session02.token == "abc"
del session02.token
del session02.token
assert session02.token is None
assert session02.events == ["assigned", "reset", "reset"]
print(session02.events)

['assigned', 'reset', 'reset']


**What this shows.** A property deleter need not literally remove a dictionary key. It implements domain semantics for `del instance.attribute`. Defining idempotence is a design decision, not an automatic rule.

---

## Problem 03 — A property exception can accidentally activate `__getattr__`

**Challenge.** Create a lazy property that is missing an internal value. Understand why raising `AttributeError` from its getter can cause a class’s fallback attribute handler to run.

**Learning path.** Build a failing example; trace the fallback; repair the data-model design with a clear missing-value exception.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Create an attribute fallback

Python invokes `__getattr__` if normal attribute lookup raises `AttributeError`. That includes the case where an existing descriptor’s getter raises `AttributeError` internally.

In [8]:
class Fallback03:
    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        return f"fallback for {name}"

    @property
    def result(self):
        return self._missing_result  # Raises AttributeError.

fallback03 = Fallback03()
print(fallback03.result)
assert fallback03.result == "fallback for result"

fallback for result


**Pause and predict.** The property internally raises `AttributeError` while reading `_missing_result`. The outer lookup then invokes `__getattr__` for the requested public name `result`, so the original problem is masked.

In [9]:
assert fallback03.result == "fallback for result"
print("The property error was masked:", fallback03.result)

The property error was masked: fallback for result


### Step 2 — Use a deliberate failure mode

When the public property represents a required value, a domain-specific exception unrelated to `AttributeError` avoids accidental fallback. Here `LookupError` communicates “not initialized.”

In [10]:
class MissingResultError(LookupError):
    pass

class Explicit03:
    def __init__(self):
        self._result = None

    def __getattr__(self, name):
        return f"fallback for {name}"

    @property
    def result(self):
        if self._result is None:
            raise MissingResultError("result not computed")
        return self._result

safe03 = Explicit03()
with expect_error(MissingResultError, "not computed"):
    _ = safe03.result
assert safe03.other == "fallback for other"

**What this shows.** Choose exceptions intentionally. `AttributeError` is appropriate when an attribute truly should appear absent (for example, so `hasattr()` returns `False`), but risky for unrelated internal bugs in getters.

---

## Problem 04 — Implement method binding with a non-data descriptor

**Challenge.** Explain why ordinary functions become bound methods when retrieved from instances. Write a tiny descriptor that replicates the binding behavior.

**Learning path.** Inspect class/instance access, implement binding, then observe shadowing.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Inspect ordinary method binding

Functions defined in classes are non-data descriptors. Access through a class yields a function; access through an instance yields a bound method whose `__self__` is that instance.

In [11]:
class Greeter04:
    def hello(self, name):
        return f"Hello {name} from {type(self).__name__}"

greeter04 = Greeter04()
assert Greeter04.hello(greeter04, "Ada") == greeter04.hello("Ada")
assert greeter04.hello.__self__ is greeter04
assert greeter04.hello.__func__ is Greeter04.hello
print(greeter04.hello("Ada"))

Hello Ada from Greeter04


### Step 2 — Implement `__get__` yourself

`types.MethodType(function, instance)` produces a bound method. Class lookup has `instance is None`, so this small educational implementation returns its raw function.

In [12]:
from types import MethodType

class BindLikeFunction04:
    def __init__(self, function):
        self.function = function

    def __get__(self, instance, owner=None):
        if instance is None:
            return self.function
        return MethodType(self.function, instance)

class Robot04:
    @BindLikeFunction04
    def identify(self, code):
        return (type(self).__name__, code)

robot04 = Robot04()
assert Robot04.identify(robot04, 7) == robot04.identify(7)
assert robot04.identify.__self__ is robot04
print(robot04.identify(7))

('Robot04', 7)


### Step 3 — Show the non-data consequence

The descriptor implements `__get__`, but neither `__set__` nor `__delete__`. An ordinary instance dictionary entry can therefore shadow it; deletion of that instance entry restores descriptor access.

In [13]:
assert not hasattr(Robot04.__dict__["identify"], "__set__")
robot04.identify = lambda code: ("patched", code)
assert robot04.identify(7) == ("patched", 7)
del robot04.identify
assert robot04.identify(7) == ("Robot04", 7)
print("Method behavior restored after deleting the instance override.")

Method behavior restored after deleting the instance override.


**What this shows.** Method binding is descriptor behavior. Our tiny wrapper is a subset of a real Python function object, which also provides signatures, introspection information, and more.

---

## Problem 05 — Instance methods vs. static and class methods

**Challenge.** Predict what receives an implicit argument when each kind of method is read through a class or instance. Investigate inherited factory methods.

**Learning path.** Build all three variants, inspect their raw wrappers, and test construction in a subclass.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Build three method flavors

`staticmethod` suppresses implicit binding. `classmethod` binds the accessing class, allowing inherited factories to construct subclasses.

In [14]:
class Factory05:
    def instance_kind(self):
        return type(self).__name__

    @staticmethod
    def parse(number):
        return int(number)

    @classmethod
    def build(cls, number):
        created = cls()
        created.number = cls.parse(number)
        return created

class SpecializedFactory05(Factory05):
    pass

item05 = SpecializedFactory05.build("12")
assert type(item05) is SpecializedFactory05
assert item05.number == 12
assert item05.instance_kind() == "SpecializedFactory05"

### Step 2 — Inspect what the class actually stores

A class dictionary holds descriptor wrappers for static methods and class methods. Reading the attributes normally resolves those wrappers into callables.

In [15]:
raw_static05 = Factory05.__dict__["parse"]
raw_class05 = Factory05.__dict__["build"]
assert isinstance(raw_static05, staticmethod)
assert isinstance(raw_class05, classmethod)
assert Factory05.parse("3") == Factory05().parse("3") == 3
assert SpecializedFactory05.build.__self__ is SpecializedFactory05
assert Factory05.build.__self__ is Factory05
print("Class-method receiver:", SpecializedFactory05.build.__self__.__name__)

Class-method receiver: SpecializedFactory05


### Step 3 — Test the difference inheritance makes

A factory hard-coded to `Factory05()` would always return the base type. `classmethod` supplies the actual accessing class as `cls` instead.

In [16]:
base05 = Factory05.build("8")
sub05 = SpecializedFactory05.build("8")
assert type(base05) is Factory05
assert type(sub05) is SpecializedFactory05
print(type(base05).__name__, type(sub05).__name__)

Factory05 SpecializedFactory05


**What this shows.** Instance methods bind objects, class methods bind classes, and static methods bind neither. All three are resolved through attribute lookup, but their binding policies differ.

---

## Problem 06 — Use the `owner` argument under inheritance

**Challenge.** Create a descriptor whose result depends on the accessing class. Distinguish the class where a descriptor was declared from the class used to access it.

**Learning path.** Implement `__get__` and `__set_name__`; compare access from a base, subclass, and subclass instance.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Make the accessing owner visible

For `SubClass.field`, the descriptor receives `instance=None` and `owner=SubClass`. For an instance read, `owner` normally refers to the type of that instance.

In [17]:
class OwnerLabel06:
    def __set_name__(self, owner, name):
        self.declared_name = name
        self.declaring_class = owner

    def __get__(self, instance, owner=None):
        if owner is None:
            owner = type(instance)
        return f"{owner.__name__}.{self.declared_name}"

class Document06:
    label = OwnerLabel06()

class Report06(Document06):
    pass

assert Document06.label == "Document06.label"
assert Report06.label == "Report06.label"
assert Report06().label == "Report06.label"
print(Document06.label, Report06.label)

Document06.label Report06.label


### Step 2 — Separate owner from declaring class

`__set_name__` ran once, during creation of `Document06`. Inheritance does not rerun it on `Report06`, even though the inherited descriptor can see the subclass through its `owner` argument.

In [18]:
descriptor06 = Document06.__dict__["label"]
assert descriptor06.declaring_class is Document06
assert descriptor06.declared_name == "label"
assert "label" not in Report06.__dict__
print("Declared in:", descriptor06.declaring_class.__name__)
print("Accessed as:", Report06.label)

Declared in: Document06
Accessed as: Report06.label


**What this shows.** `owner` answers “which class is used for this access?” while `__set_name__` answers “where and under what name was this descriptor installed?” They are different questions.

---

## Problem 07 — Avoid aliasing stateful descriptor metadata

**Challenge.** Reuse one descriptor under two names and discover why a simplistic `__set_name__` design can go wrong. Repair the binding design.

**Learning path.** Trigger the alias bug, use distinct descriptor objects, and check the inheritance case.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Trigger aliasing

During class creation, Python calls `__set_name__` once for each name. One shared descriptor with one `name` field gets its metadata overwritten at the second binding.

In [19]:
class NameOnly07:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        return self if instance is None else self.name

shared07 = NameOnly07()
class Aliased07:
    left = shared07
    right = shared07

assert Aliased07().left == "right"
assert Aliased07().right == "right"
print("Both aliases now claim the name:", Aliased07().left)

Both aliases now claim the name: right


### Step 2 — Create one descriptor per independent field

A one-name descriptor is best treated as a one-binding object. This implementation avoids guessing which source spelling triggered access.

In [20]:
class Independent07:
    left = NameOnly07()
    right = NameOnly07()

record07 = Independent07()
assert record07.left == "left"
assert record07.right == "right"
assert Independent07.__dict__["left"] is not Independent07.__dict__["right"]
print(record07.left, record07.right)

left right


### Step 3 — Understand the inheritance exception

An inherited descriptor retains its original name: inheritance does not itself create a new alias. The existing binding can work correctly across many subclasses.

In [21]:
class Child07(Independent07):
    pass

assert Child07().left == "left"
assert Child07().right == "right"
print("Inherited bindings retain their original names.")

Inherited bindings retain their original names.


**What this shows.** Sophisticated descriptors may track metadata by owner, but the arguments of `__get__` do not include the *attribute name used in the expression*. Using distinct descriptors is the simplest robust answer for separate fields.

---

## Problem 08 — Defensive property reads for mutable state

**Challenge.** A property validates list assignments but returns the underlying mutable list. Show how an external caller breaks the invariant without using the setter; repair the API.

**Learning path.** Demonstrate a leak, store immutable snapshots, and verify failures preserve existing values.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Demonstrate the leaky getter

An underscore is a naming convention, not access control. Returning `_scores` exposes a mutable alias that clients can modify without invoking the setter.

In [22]:
class LeakyScores08:
    def __init__(self):
        self._scores = []

    @property
    def scores(self):
        return self._scores

    @scores.setter
    def scores(self, items):
        if any(type(item) is not int or item < 0 for item in items):
            raise ValueError("scores must be nonnegative integers")
        self._scores = list(items)

leaky08 = LeakyScores08()
leaky08.scores = [3]
leaky08.scores.append(-99)  # Does not call the setter.
assert leaky08.scores == [3, -99]
print("Invariant broken:", leaky08.scores)

Invariant broken: [3, -99]


### Step 2 — Store immutable data and return an immutable value

Convert incoming data into a tuple before validation and assignment. This also breaks aliases with the caller’s input list.

In [23]:
class Scores08:
    def __init__(self, items=()):
        self.scores = items

    @property
    def scores(self):
        return self._scores

    @scores.setter
    def scores(self, items):
        snapshot = tuple(items)
        if any(type(item) is not int or item < 0 for item in snapshot):
            raise ValueError("scores must be nonnegative integers")
        self._scores = snapshot

original08 = [2, 4]
safe08 = Scores08(original08)
original08.append(-99)
assert safe08.scores == (2, 4)
with expect_error(AttributeError):
    safe08.scores.append(10)

### Step 3 — Verify rejected assignment preserves old state

The candidate tuple is validated before replacing `_scores`. Failure leaves the previous tuple untouched.

In [24]:
before08 = safe08.scores
with expect_error(ValueError):
    safe08.scores = [1, -1]
assert safe08.scores is before08
safe08.scores = [5, 10]
assert safe08.scores == (5, 10)
print(safe08.scores)

(5, 10)


**What this shows.** A tuple only prevents mutation of its structure. If elements themselves are mutable, deeper copying or a domain-specific immutable representation may be needed. Property validation is not automatically deep immutability.

---

## Problem 09 — Dependency-aware invalidation for cached properties

**Challenge.** An expensive derived value depends on quantity and price. Cache it, but ensure both input setters make stale results impossible.

**Learning path.** Track a version, cache a result with its version, and verify recomputation after each dependency changes.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Define versioned managed state

Successful changes increment a shared `_version`; rejected or no-op assignments leave it unchanged. The version represents which dependency state the cache was computed from.

In [25]:
from math import isfinite

class Pricing09:
    def __init__(self, quantity, unit_price):
        self._version = 0
        self._calculation_count = 0
        self._cache = None
        self._quantity = 0
        self._unit_price = 0.0
        self.quantity = quantity
        self.unit_price = unit_price

    @property
    def quantity(self):
        return self._quantity

    @quantity.setter
    def quantity(self, value):
        if type(value) is not int or value < 0:
            raise ValueError("quantity must be a nonnegative int")
        if value != self._quantity:
            self._quantity = value
            self._version += 1

    @property
    def unit_price(self):
        return self._unit_price

    @unit_price.setter
    def unit_price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("unit_price must be numeric")
        number = float(value)
        if not isfinite(number) or number < 0:
            raise ValueError("unit_price must be finite and nonnegative")
        if number != self._unit_price:
            self._unit_price = number
            self._version += 1

### Step 2 — Compute lazily using the version

This cache lives at the private name `_cache`, rather than shadowing the public descriptor. `total` remains a read-only built-in property.

In [26]:
def calculate_total09(self):
    cached = self._cache
    if cached is None or cached[0] != self._version:
        self._calculation_count += 1
        answer = self.quantity * self.unit_price
        self._cache = (self._version, answer)
    return self._cache[1]

Pricing09.total = property(calculate_total09, doc="Cached extended price.")
price09 = Pricing09(2, 12.5)
assert price09.total == price09.total == 25.0
assert price09._calculation_count == 1
print("Calculations:", price09._calculation_count)

Calculations: 1


### Step 3 — Test invalidation and rejected writes

Assignments changing either input force a recalculation on the next read. An invalid assignment must not modify the version or cached result.

In [27]:
price09.quantity = 3
assert price09.total == 37.5
price09.unit_price = 10
assert price09.total == 30.0
assert price09._calculation_count == 3
old_version09 = price09._version
with expect_error(ValueError):
    price09.quantity = -1
assert price09._version == old_version09
assert price09.total == 30.0
assert price09._calculation_count == 3
print("Cache invalidation verified.")

Cache invalidation verified.


**What this shows.** Versioning helps when multiple derived values depend on common state. This educational version is single-threaded: concurrency requires coordination between state changes and computation.

---

## Problem 10 — Inspect descriptors without triggering getter side effects

**Challenge.** An introspection utility should describe a property without evaluating it. Compare normal lookup with `inspect.getattr_static`.

**Learning path.** Instrument a getter, inspect its raw property, and build a safe metadata reporter.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Make getter side effects observable

`getattr(instance, name)` performs full attribute lookup and may execute getter code. Here getter execution changes a counter.

In [28]:
class Sensor10:
    def __init__(self):
        self.read_count = 0

    @property
    def pressure(self):
        """Pressure in arbitrary units; read is instrumented."""
        self.read_count += 1
        return 101

sensor10 = Sensor10()
assert getattr(sensor10, "pressure") == 101
assert sensor10.read_count == 1
print("Getter calls:", sensor10.read_count)

Getter calls: 1


### Step 2 — Get the raw descriptor

`inspect.getattr_static` avoids dynamic attribute lookup and descriptor invocation. In this simple class, it returns the actual `property` object.

In [29]:
import inspect

raw10 = inspect.getattr_static(sensor10, "pressure")
assert isinstance(raw10, property)
assert raw10.fget.__name__ == "pressure"
assert "instrumented" in raw10.__doc__
assert sensor10.read_count == 1
print("Inspected:", raw10.fget.__name__)

Inspected: pressure


### Step 3 — Build a property metadata reporter

The reporter never invokes the getter. It reads `fget`, `fset`, and `fdel` from the raw property descriptor to describe its capabilities.

In [30]:
def describe_property10(obj, name):
    raw = inspect.getattr_static(obj, name)
    if not isinstance(raw, property):
        raise TypeError(f"{name} is not a built-in property")
    return {
        "name": name,
        "readable": raw.fget is not None,
        "writable": raw.fset is not None,
        "deletable": raw.fdel is not None,
        "doc": raw.__doc__,
    }

report10 = describe_property10(sensor10, "pressure")
assert report10["readable"] and not report10["writable"]
assert sensor10.read_count == 1
print(report10)

{'name': 'pressure', 'readable': True, 'writable': False, 'deletable': False, 'doc': 'Pressure in arbitrary units; read is instrumented.'}


**What this shows.** Static inspection is handy for documentation and serializers that should not trigger getters. It does not make arbitrary returned objects safe to execute; calling `raw.fget(sensor10)` would still run code.

---

## Problem 11 — A slotted dataclass with a validated property

**Challenge.** Use dataclass-generated initialization while keeping a public balance governed by a property. Avoid a dataclass field/property collision.

**Learning path.** Use `InitVar`, a private slotted field, `__post_init__`, and invariant tests.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Keep constructor input separate from the property

`opening_balance` is an `InitVar`: it is passed to `__post_init__` instead of stored as an ordinary dataclass field. The public property `balance` writes to a declared private slot.

In [31]:
from dataclasses import InitVar, dataclass, field, fields
from decimal import Decimal, InvalidOperation

@dataclass(slots=True)
class Wallet11:
    opening_balance: InitVar[str]
    owner: str
    _balance: Decimal = field(init=False, repr=False)

    def __post_init__(self, opening_balance):
        self.balance = opening_balance

    @property
    def balance(self) -> Decimal:
        return self._balance

    @balance.setter
    def balance(self, value) -> None:
        try:
            amount = Decimal(value)
        except (InvalidOperation, TypeError, ValueError) as exc:
            raise ValueError("balance must be a decimal amount") from exc
        if not amount.is_finite() or amount < 0:
            raise ValueError("balance must be finite and nonnegative")
        self._balance = amount

### Step 2 — Check constructor, property, and slots

With `slots=True` and no dictionary-bearing base, the instance has no `__dict__`. Our property instead uses the declared `_balance` slot.

In [32]:
wallet11 = Wallet11("12.50", owner="Ada")
assert wallet11.balance == Decimal("12.50")
assert not hasattr(wallet11, "__dict__")
assert "opening_balance" not in {entry.name for entry in fields(Wallet11)}
assert isinstance(Wallet11.__dict__["balance"], property)
print(wallet11, "balance:", wallet11.balance)

Wallet11(owner='Ada') balance: 12.50


### Step 3 — Test both initialization and updates

The constructor and later assignments reach the *same* validation code. Decimal strings avoid introducing a binary float before constructing `Decimal`.

In [33]:
with expect_error(ValueError, "nonnegative"):
    Wallet11("-1", owner="Ada")
with expect_error(ValueError, "finite"):
    wallet11.balance = "Infinity"
assert wallet11.balance == Decimal("12.50")
wallet11.balance = "20.75"
assert wallet11.balance == Decimal("20.75")
print("Slotted dataclass property passed.")

Slotted dataclass property passed.


**What this shows.** The key design is different names for generated constructor arguments and public descriptors. Defining a dataclass field with the same name as a property can create unexpected defaults or collisions.

---

## Problem 12 — Collect descriptor schemas across inheritance

**Challenge.** Build a reusable field descriptor plus a per-class registry. Subclasses must inherit fields and add new ones without mutating parent registries.

**Learning path.** Create a field, merge registries in `__init_subclass__`, then test overrides and instance isolation.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Build a field with instance-local storage

A field stores immutable-ish configuration on the descriptor and validated values on each instance. The validator returns the cleaned value; it may raise before storage changes.

In [34]:
class SchemaField12:
    def __init__(self, clean):
        self.clean = clean

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"_schema_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError as exc:
            raise AttributeError(f"{self.name} is not set") from exc

    def __set__(self, instance, value):
        cleaned = self.clean(value)
        instance.__dict__[self.key] = cleaned

### Step 2 — Build an inherited registry with a class hook

`__init_subclass__` is called on subclass creation. First merge ancestor registries in reverse MRO order, then account for descriptors and ordinary overrides introduced by this class.

In [35]:
class SchemaModel12:
    __schema__ = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        collected = {}
        for ancestor in reversed(cls.__mro__[1:]):
            collected.update(getattr(ancestor, "__schema__", {}))
        for name, candidate in cls.__dict__.items():
            if isinstance(candidate, SchemaField12):
                collected[name] = candidate
            elif name in collected:
                collected.pop(name)  # Explicit override removes the field.
        cls.__schema__ = collected

    def __init__(self, **values):
        unknown = set(values) - set(type(self).__schema__)
        if unknown:
            raise TypeError(f"unknown fields: {sorted(unknown)}")
        for name, value in values.items():
            setattr(self, name, value)

### Step 3 — Make a base schema and an extended schema

Each class gets a distinct schema mapping, but inherited descriptor objects can remain shared because their original name binding and per-instance value storage are stable.

In [36]:
def clean_positive12(value):
    if type(value) is not int or value <= 0:
        raise ValueError("must be a positive int")
    return value

def clean_text12(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("must be nonempty text")
    return value.strip()

class Item12(SchemaModel12):
    item_id = SchemaField12(clean_positive12)

class LabeledItem12(Item12):
    label = SchemaField12(clean_text12)

assert list(Item12.__schema__) == ["item_id"]
assert list(LabeledItem12.__schema__) == ["item_id", "label"]
assert Item12.__schema__ is not LabeledItem12.__schema__
print("base:", list(Item12.__schema__), "child:", list(LabeledItem12.__schema__))

base: ['item_id'] child: ['item_id', 'label']


### Step 4 — Test independence, invalid values, and overrides

The subclass stores values independently for each instance. Ordinary class-level override removes that member from the *registry* while leaving the ancestor untouched.

In [37]:
item12 = Item12(item_id=1)
labeled12 = LabeledItem12(item_id=2, label="  blue  ")
assert (item12.item_id, labeled12.item_id, labeled12.label) == (1, 2, "blue")
with expect_error(ValueError):
    labeled12.item_id = 0
assert labeled12.item_id == 2
with expect_error(TypeError):
    Item12(unexpected=3)

class Override12(Item12):
    item_id = "replaced"

assert "item_id" not in Override12.__schema__
assert "item_id" in Item12.__schema__
print("Schema inheritance and overrides passed.")

Schema inheritance and overrides passed.


**What this shows.** This example is an educational registry rather than a complete validation framework. A production system must specify required fields, multiple inheritance policy, serialization, and the precise meaning of incompatible overrides.

---

## Problem 13 — A locked property does not make `x += 1` atomic

**Challenge.** Explain the read/modify/write race on a managed counter. Supply an atomic increment API and demonstrate the issue without flaky scheduling tests.

**Learning path.** Lock individual accesses; manually interleave two readers and writers; use one locked method instead.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Implement synchronized accessors and an explicit operation

The lock protects each getter and setter. But `counter.value += 1` consists of a read, arithmetic, and a separate write. Only a method covering the *whole* sequence makes increment atomic.

In [38]:
from threading import RLock

class Counter13:
    def __init__(self):
        self._lock = RLock()
        self._value = 0

    @property
    def value(self):
        with self._lock:
            return self._value

    @value.setter
    def value(self, candidate):
        if type(candidate) is not int or candidate < 0:
            raise ValueError("value must be a nonnegative int")
        with self._lock:
            self._value = candidate

    def increment(self, amount=1):
        if type(amount) is not int or amount < 0:
            raise ValueError("amount must be a nonnegative int")
        with self._lock:
            self._value += amount
            return self._value

### Step 2 — Simulate a lost update without threads

Suppose two workers both read the old value before either writes. Interleave those four operations manually to expose the logical error deterministically.

In [39]:
unsafe13 = Counter13()
a_read13 = unsafe13.value
b_read13 = unsafe13.value
unsafe13.value = a_read13 + 1
unsafe13.value = b_read13 + 1
assert unsafe13.value == 1  # Two attempts, one increment lost.
print("Unsafe interleaving produces:", unsafe13.value)

Unsafe interleaving produces: 1


### Step 3 — Use a single atomic operation

The API hides intermediate state: callers ask for an increment, not a read followed by a write. The method’s lock covers the entire critical section.

In [40]:
safe13 = Counter13()
assert safe13.increment() == 1
assert safe13.increment() == 2
with expect_error(ValueError):
    safe13.increment(-1)
assert safe13.value == 2
print("Atomic increments:", safe13.value)

Atomic increments: 2


**What this shows.** A lock protects a code region, not an attribute name in the abstract. Thread-safe individual accessors do not automatically create a transaction from multiple operations.

---

## Problem 14 — Capstone: lazy resource management with cleanup

**Challenge.** Design a data descriptor that opens a distinct resource lazily for each instance, forbids direct replacement, closes on deletion, and recovers correctly from a factory failure.

**Learning path.** Define lifecycle states, build a fake resource, implement the descriptor, improve errors, and test independence and failure recovery.

Try to predict the behavior at each checkpoint before executing the following cells. The worked solution is built in stages rather than presented as a single finished program.

### Step 1 — Specify the lifecycle and create a fake resource

We define unopened and open states. The first read opens a resource; repeated reads reuse it. Deletion closes the resource and returns to unopened. Factory failure must not store any incomplete value.

In [41]:
class FakeConnection14:
    def __init__(self, identifier):
        self.identifier = identifier
        self.closed = False

    def close(self):
        self.closed = True

    def __repr__(self):
        return f"FakeConnection14({self.identifier!r}, closed={self.closed})"

connection14 = FakeConnection14("example")
assert not connection14.closed
connection14.close()
assert connection14.closed
print(connection14)

FakeConnection14('example', closed=True)


### Step 2 — Implement the first descriptor draft

The descriptor itself stores a factory and a storage-key name. The *resource value* is kept in each instance dictionary. Implementing `__set__` makes the descriptor a data descriptor even though it rejects replacement.

In [42]:
class LazyResource14:
    def __init__(self, factory):
        self.factory = factory

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_key = f"_lazy_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        data = instance.__dict__
        if self.storage_key not in data:
            created = self.factory(instance)  # Failure occurs before caching.
            data[self.storage_key] = created
        return data[self.storage_key]

    def __set__(self, instance, value):
        raise AttributeError(f"{self.name} is read-only; delete it to reset")

    def __delete__(self, instance):
        data = instance.__dict__
        resource = data[self.storage_key]
        resource.close()
        del data[self.storage_key]

**Refinement checkpoint.** Our first draft of `__delete__` raises a low-level `KeyError` on an unopened resource. Let us choose a deliberate contract: deleting an unopened resource raises `AttributeError`; if `close()` raises, preserve the cached reference so a caller can diagnose or retry. We make a refined descriptor class **before** binding it to a real owner.

### Step 3 — Refine cleanup semantics and create an owner

Using a fresh refined descriptor means `__set_name__` will be called automatically during `Service14` class creation.

In [43]:
class ManagedResource14(LazyResource14):
    def __delete__(self, instance):
        data = instance.__dict__
        if self.storage_key not in data:
            raise AttributeError(f"{self.name} is not open")
        resource = data[self.storage_key]
        resource.close()  # If this raises, keep the cached reference.
        del data[self.storage_key]

class Service14:
    connection = ManagedResource14(lambda instance: instance._open_connection())

    def __init__(self, identifier):
        self.identifier = identifier
        self.open_count = 0
        self.fail_next = False

    def _open_connection(self):
        self.open_count += 1
        if self.fail_next:
            self.fail_next = False
            raise OSError("simulated connection failure")
        return FakeConnection14(self.identifier)

### Step 4 — Test per-instance state and cleanup

Repeated reads share one resource per owner, not one resource across owners. `del` closes one owner’s resource without altering any other owner.

In [44]:
alpha14, beta14 = Service14("alpha"), Service14("beta")
assert Service14.connection is Service14.__dict__["connection"]
a14 = alpha14.connection
b14 = beta14.connection
assert a14 is alpha14.connection
assert a14 is not b14
assert (alpha14.open_count, beta14.open_count) == (1, 1)
with expect_error(AttributeError, "read-only"):
    alpha14.connection = FakeConnection14("injected")
del alpha14.connection
assert a14.closed and not b14.closed
with expect_error(AttributeError, "not open"):
    del alpha14.connection
new_alpha14 = alpha14.connection
assert new_alpha14 is not a14
assert alpha14.open_count == 2
print("Lifecycle verified for two independent owners.")

Lifecycle verified for two independent owners.


### Step 5 — Prove failed opening does not poison the cache

The factory raises before the instance dictionary is updated. The following read may retry successfully; the cached value must then be closed on deletion.

In [45]:
unstable14 = Service14("unstable")
unstable14.fail_next = True
with expect_error(OSError, "simulated"):
    _ = unstable14.connection
assert "_lazy_connection" not in unstable14.__dict__
recovered14 = unstable14.connection
assert not recovered14.closed
assert unstable14.open_count == 2
del unstable14.connection
assert recovered14.closed
print("Failure recovery verified.")

Failure recovery verified.


**What this shows.** Resource descriptors are convenient for lifecycle management, but real connection pools also need explicit concurrency, shutdown ordering, context-manager semantics, and a policy for failures during `close()`. This capstone intentionally models only a single-threaded instance lifecycle.

---

## Final synthesis — reason from the protocol

The problems above illustrate concepts that simple `@property` examples often hide:

1. **Property composition creates replacement objects.** Adding a setter or deleter produces a new property configuration.
2. **Access has consequences.** Attribute reads can invoke getters, trigger `__getattr__`, and change state. Use static introspection when execution is undesirable.
3. **Binding is descriptor behavior.** Ordinary functions, static methods, and class methods have distinct receiver policies.
4. **One descriptor can serve many objects, but metadata matters.** Separate owner metadata from accessing class; avoid installing a one-name descriptor under unrelated aliases.
5. **A validated setter is not a complete data model.** External mutable aliases, stale derived caches, and compound thread operations need their own solutions.
6. **Deletion expresses a lifecycle contract.** Reset and cleanup are both legitimate meanings, provided failure behavior is explicit.

**Further independent challenge:** combine safe static inspection from Problem 10 with the inherited schema registry in Problem 12 to generate field documentation without reading a single managed value. Which kinds of user-provided descriptor metadata should you still treat as untrusted?